Note: Write your code in the code cells, and your responses in markdown. 
Run the entire script and display the outputs of your code. 

Due: **11:59PM Central Time on Monday, 11/03**. Upload both your code (.ipynb) and responses (html or pdf) to Canvas by then. 

In [ ]:
# Packages you might need: (pip install ... if you don't have them)
import pandas as pd # for data manipulation
import os  # for setting directory 
print(os.getcwd())
# os.chdir() # input your personal directory where the dataset is saved
import statsmodels.formula.api as smf # for OLS regressions
import numpy as np  # to work with arrays (vectors/matrices)

# Princeton Twins Data
In this problem set we will fit a few models to the Princeton Twins Survey data. The data set is called twins.csv. The variables are:
- famid = family id variable
- t=1,2 for twin #1 or twin #2
- age = age (some observations have coded in part year values)
- educ = education
- oeduc = education of twin
- lw = log wage
- married = dummy 1 if married 0 if not
- omarried = dummy if twin is married
- female = 1 if female
- ofemale = 1 if twin is female.
- exp = “labor market experience” = age - educ - 6
- oexp = experience of twin

NOTE: all twins are one of two identical twins in the data set. So female=ofemale in all cases.

In [ ]:
# Load dataset (using pandas "pd")
twins = pd.read_csv("twins.csv") # add your own directory if necessary
print(twins.head(6))

# verify the data is unique at (famid, t) level, where t = 1,2 for twin #1 or #2
assert 0==twins[['famid','t']].duplicated().sum()

# verify twins have the same sex: 
assert (twins['female']==twins['ofemale']).all()

## 1. OLS
Estimate a simple model relating log wages to: education, experience, experience-squared, married status, and female. 

In [ ]:
# controls: add experienced squared 
twins['exp2'] = twins['exp']**2 

# OLS of log wage on controls: 
model = smf.ols(f"lw ~ educ + exp + exp2 + married + female", data=twins).fit(cov_type='HC1')
print(model.summary())

## 2. Separate models for men and women
Fit the same model separately for men and women. Does the model look different?

In [ ]:
# fit the same model for men: 
model_m = smf.ols(f"lw ~ educ + exp + exp2 + married", data=twins.loc[twins['female']==0]).fit(cov_type='HC1')
print("Males \n", model_m.summary())

In [ ]:
# for women: 
model_f = smf.ols(f"lw ~ educ + exp + exp2 + married", data=twins.loc[twins['female']==1]).fit(cov_type='HC1')
print("Females \n", model_f.summary())

The OLS estimate of return to education is higher for females. Married females earn 4 log-point lower wages than unmarried females conditional on other observables. In contrast, married males earn 20 log-point higher wages than married males on average, conditional on controls for education and experience. 

## 3. Add mean family marriage rate
Construct the mean family marriage rate for each person in the data set (i.e., the fraction of the twins that is married, which can be 0, 1/2, or 1).

In [ ]:
fam = twins.groupby(['famid'])['married'].mean().reset_index(drop=False)
fam.columns=['famid','mean_married']
# merge into twins:
twins = pd.merge(twins, fam, on=['famid'], how='left')
print(twins['mean_married'].value_counts(dropna=False))

### (a) 
Verify that when you regress marriage of a twin on the average fraction of the siblings who are married, you get a coefficient of 1.

In [ ]:
model0 = smf.ols(f"married~ mean_married", data=twins).fit(cov_type='HC1')
print(model0.summary())

### (b) 
Add mean fraction of siblings married to your gender-specific wage models from part 2. How does the addition of this variable affect the estimated coefficient on marriage. Give an interpretation of the patterns and how they differ between men and women. 

In [ ]:
# load coef on married from previous models: 
coef_married = {'male': model_m.params['married'].item(), 'female': model_f.params['married'].item()}
print(coef_married)

In [ ]:
# fit the revised model for men: 
model2_m = smf.ols(f"lw ~ educ + exp + exp2 + married + mean_married", data=twins.loc[twins['female']==0]).fit(cov_type='HC1')
print("Males \n", model2_m.summary())
# for women: 
model2_f = smf.ols(f"lw ~ educ + exp + exp2 + married  + mean_married", data=twins.loc[twins['female']==1]).fit(cov_type='HC1')
print("Females \n", model2_f.summary())

In [ ]:
# update coef on married from previous models: 
coef_married.update({'male_w_mean': model2_m.params['married'].item(), 'female_w_mean': model2_f.params['married'].item()})
print(coef_married)

When we control for the mean marital status in each family, by Frisch-Waugh, the revised regression identifies the effect of marriage based on variation in marital status *within* families. We can interpret the coefficient on marriage as the effect of marriage on earnings holding family-specific factors that matter for marriage fixed. 

That being said, this is not the within-family estimate yet because there could be other family-specific factors that matter for marriage but not taken into account by the mean family marriage rate. Question 4 shows 3 ways to get the within estimates. 


### (c)
Instead of controlling for the mean family marriage rate, estimate the gender-specific wage models from part 2 with a control for the other twin's marriage status (variable “omarried”). Compare the regression coefficients on (married, omarried) with the coefficient on (married, mean married) in (b). 

In [ ]:
# fit the revised model for men: 
model3_m = smf.ols(f"lw ~ educ + exp + exp2 + married + omarried", data=twins.loc[twins['female']==0]).fit(cov_type='HC1')
print("Males \n", model3_m.params[['married','omarried']])
# for women: 
model3_f = smf.ols(f"lw ~ educ + exp + exp2 + married  + omarried", data=twins.loc[twins['female']==1]).fit(cov_type='HC1')
print("Females \n", model3_f.params[['married','omarried']])

Since $t=1,2$ in each family, instead of the mean we can also control for the characteristics of the other twin. But we need to compute the difference between the coefficient on "married" and the coefficient on "omarried" (the other twin) in order to retrieve the "within" effect of marriage on earnings in (b).

In [ ]:
print("compare with coefficients with mean_married:\n",
      'male:', coef_married['male_w_mean'],
      'female:', coef_married['female_w_mean'])

print('males (from model3): \n', model3_m.params['married'].item() - model3_m.params['omarried'])
print('females (from model3): \n', model3_f.params['married'] - model3_f.params['omarried'])

## 4. Three ways to get the within estimator: 
In lecture 8 we show three ways to get the within estimator - de-meaned relative to the mean, fixed effects, and control function. Let's run 3 regressions for **male** twins. 

### (a) 
De-mean variables relative to the mean in each family: replace $x_{it}$ by $x_{it}-\bar{x}_{i}$ where $x_{it}$ includes education, experience, experience-squared, and married status, and replace $y_{it}$ by $y_{it}-\bar{y}_{i}$. Fit a regression of $(y_{it}-\bar{y}_{i})$ on $x_{it}-\bar{x}_{i}$.

In [ ]:
print(twins.columns)
select= ['lw','educ','exp','exp2','married']
print(twins[select])

In [ ]:
keys = ['famid','t'] # twins data unique at (family, twin) level
select= ['lw','educ','exp','exp2','married']
# focus on families with male twins:
fam_means = twins.loc[twins['female']==0].groupby(['famid'])[select].mean().reset_index(drop=False)
fam_means.columns=['famid'] + [f'mean_{x}' for x in select]

# merge fam_means with twins (note you may have defined mean marriage rate already) by famid
male_twins = pd.merge(twins.loc[twins['female']==0, keys+select], fam_means, on=['famid'],how='left')

# compute the demeaned x and y: 
for z in select:
    male_twins[f'd_{z}'] = male_twins[z] - male_twins[f'mean_{z}'] 

# note: d_exp = - d_educ. exp is defined as age - educ - 6. Within each family, twins are of the same age, d_exp = exp - (age  - mean_educ -6) = - educ + mean_educ = - d_educ
print(male_twins[['d_exp','d_educ']].corr())
assert (abs(male_twins['d_exp'] + male_twins['d_educ'])<1e-12).all()

In [ ]:
# within estimator: 
model_within = smf.ols(f"d_lw ~ d_educ  + d_exp2 + d_married", data=male_twins).fit(cov_type='HC1')
print("Males - Within estimator \n", model_within.summary())
coef_within = model_within.params[['d_educ','d_exp2','d_married']]

# check we should get the same coef on covariates with lw as outcome:
model2_within =  smf.ols(f"lw ~ d_educ + d_exp2 + d_married", data=male_twins).fit(cov_type='HC1')
print("Males \n", model2_within.summary())

### (b) Fixed Effects
Fit a regression of y_{it} on x_{it} and family fixed effects. 
- *Hint if you use PanelOLS from linearmodels, make sure to drop collinear variables before adding “entityeffects”. For example, educ and exp are collinear with each other conditional on family fixed effects. Why? See how experience is defined, and note twins have the same age.*

In [ ]:
from linearmodels import PanelOLS

# This automatically absorbs fixed effects without showing them
model_FE= PanelOLS.from_formula("lw ~ 1 + educ  + exp2 + married   + EntityEffects", 
                               data=male_twins.set_index(['famid','t'])).fit(cov_type='clustered', cluster_entity=True) 
print(model_FE)
coef_FE = model_FE.params[['educ','exp2','married']]


### (c) Control Function
Fit a regression of $y_{it}$ on $x_{it}$ and means $\bar{x}_{i}$ in each family. Verify if the coefficients on $x_{it}$ are the same as in (a) and (b). 

In [ ]:
model_cf = smf.ols(f"lw ~ educ + exp2 + married + mean_educ + mean_exp2 + mean_married", data=male_twins).fit(cov_type='HC1')
print(model_cf.summary())
coef_cf = model_cf.params[['educ','exp2','married']]

In [ ]:
# Verify the coefficients are the same across 3 approaches: 
print("coef in model (a)-within: \n", coef_within)
print("coef in model (b)-fixed effects: \n", coef_FE)
print("coef in model (c)-control function: \n", coef_cf)